In [1]:
import watermark
pkg_versions = watermark.watermark(
    packages="requests,pandas,tqdm")
print(pkg_versions)

requests: 2.32.5
pandas  : 1.5.3
tqdm    : 4.67.1



In [2]:
import os
import requests
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm

load_dotenv()
API_KEY = os.getenv("API_KEY")
BASE_URL = "https://apis.data.go.kr"

# 행정안전부_행정표준코드_법정동코드

In [3]:
URL = f"{BASE_URL}/1741000/StanReginCd/getStanReginCdList"
params = {
    "serviceKey":API_KEY,
    "numOfRows": 1000,
    "pageNo": 1,
    "flag":"Y",
    "locatadd_nm":"부산광역시",
    "type":"json"
}
res = requests.get(URL, params= params)
data = res.json()
station_df = pd.DataFrame(data["StanReginCd"][1]["row"])
station_df["signguCode"] = (
    station_df["sido_cd"].astype(str).str.zfill(2)
    + station_df["sgg_cd"].astype(str).str.zfill(3)
)
busan_df = station_df[
    station_df["locallow_nm"].str.contains(
        r".*(?:구|군)$", na=False
    )
].reset_index(drop=True)

In [4]:
busan_df

,region_cd,sido_cd,sgg_cd,umd_cd,ri_cd,locatjumin_cd,locatjijuk_cd,locatadd_nm,locat_order,locat_rm,locathigh_cd,locallow_nm,adpt_de,signguCode
0,2611000000,26,110,000,00,2611000000,2611000000,부산광역시 중구,1,,2600000000,중구,,26110
1,2614000000,26,140,000,00,2614000000,2614000000,부산광역시 서구,2,,2600000000,서구,,26140
2,2617000000,26,170,000,00,2617000000,2617000000,부산광역시 동구,3,,2600000000,동구,,26170
3,2620000000,26,200,000,00,2620000000,2620000000,부산광역시 영도구,4,,2600000000,영도구,,26200
4,2623000000,26,230,000,00,2623000000,2623000000,부산광역시 부산진구,5,,2600000000,부산진구,,26230
5,2626000000,26,260,000,00,2626000000,2626000000,부산광역시 동래구,6,,2600000000,동래구,,26260
6,2629000000,26,290,000,00,2629000000,2629000000,부산광역시 남구,7,,2600000000,남구,,26290
7,2632000000,26,320,000,00,2632000000,2632000000,부산광역시 북구,8,,2600000000,북구,,26320
8,2635000000,26,350,000,00,2635000000,2635000000,부산광역시 해운대구,9,,2600000000,해운대구,,26350
9,2638000000,26,380,000,00,2638000000,2638000000,부산광역시 사하구,10,,2600000000,사하구,,26380


In [5]:
busan_codes = busan_df["signguCode"].unique()

# 한국관광공사_국문 관광정보 서비스_GW
- 지역기반 관광정보 조회

In [6]:
URL = f"{BASE_URL}/B551011/KorService2/areaBasedList2"

In [7]:
params = {
    "serviceKey":API_KEY,
    "numOfRows": 1,
    "pageNo": 1,
    "MobileOS":"IOS",
    "MobileApp":"AppTest",
    "lDongRegnCd":26,
    # "lDongSignguCd":busan_codes[0],
    # "modifiedtime":"202609",
    "_type":"json"
}

In [8]:
res = requests.get(URL, params= params)
data = res.json()
total_count = data["response"]["body"]["totalCount"]
total_count

2221

In [9]:
num_of_rows = 1000
params.update({"numOfRows":num_of_rows})

dfs = list()
for page in tqdm(range(1, total_count//num_of_rows + 2)):
    params.update({"pageNo":page})
    res = requests.get(URL, params= params)
    data = res.json()
    df = pd.DataFrame(data["response"]["body"]["items"]["item"])
    dfs.append(df)

100%|██████████| 3/3 [00:00<00:00,  3.03it/s]


In [15]:
df = pd.concat(dfs,ignore_index=True)
df["signguCode"] = (
    df["lDongRegnCd"].astype(str).str.zfill(2)
    + df["lDongSignguCd"].astype(str).str.zfill(3)
)
df.head()

,addr1,addr2,areacode,cat1,cat2,cat3,contentid,contenttypeid,createdtime,firstimage,...,sigungucode,tel,title,zipcode,lDongRegnCd,lDongSignguCd,lclsSystm1,lclsSystm2,lclsSystm3,signguCode
0,부산광역시 부산진구 중앙번영로 (6),,,,,,2805408,39,20220125140006,,...,,,가가와,47361,26,230,FD,FD02,FD020200,26230
1,부산광역시 부산진구 가야대로 779 (부전동),,,,,,2930927,38,20221030154457,https://tong.visitkorea.or.kr/cms/resource/90/...,...,,,가까운약국,47257,26,230,SH,SH04,SH040300,26230
2,부산광역시 강서구 서천로42번길 351 (천성동),,,,,,2715601,12,20210508004846,https://tong.visitkorea.or.kr/cms/resource/69/...,...,,,가덕도,46770,26,440,NA,NA02,NA020500,26440
3,부산광역시 강서구 외양포로 10,,6,A01,A0101,A01011600,129156,12,20060719090000,http://tong.visitkorea.or.kr/cms/resource/81/3...,...,1,,가덕도 등대,46771,26,440,VE,VE01,VE010800,26440
4,부산광역시 강서구 천성동,,6,A01,A0101,A01010400,2726843,12,20210723203306,,...,1,,가덕도 연대봉,,26,440,NA,NA01,NA010100,26440


In [16]:
df.shape

(2221, 26)

In [18]:
merged_df = pd.merge(busan_df[["locallow_nm","signguCode"]], df, how="outer", on = "signguCode")
merged_df.shape

(2221, 27)

In [21]:
merged_df.to_csv("국문 관광정보 서비스_GW - 지역기반 관광정보 조회.csv",index=False)

# 한국관광공사_관광지별 연관 관광지 정보(미진행)

In [6]:
URL = f"{BASE_URL}/B551011/TarRlteTarService1/areaBasedList1"

In [15]:
params = {
    "serviceKey":API_KEY,
    "numOfRows": 1,
    "pageNo": 1,
    "MobileOS":"IOS",
    "MobileApp":"AppTest",
    "baseYm":"202608",
    "areaCd":"26",
    "signguCd":busan_codes[0],
    "_type":"json"
}

In [16]:
res = requests.get(URL, params= params)
data = res.json()
total_count = data["response"]["body"]["totalCount"]

In [17]:
data

{'response': {'header': {'resultCode': '0000', 'resultMsg': 'OK'},
  'body': {'items': {'item': [{'baseYm': '202608',
      'tAtsCd': '9dc07e5a6565aa80ebc457a0b3dd5c2c',
      'tAtsNm': '40계단거리',
      'areaCd': '26',
      'areaNm': '부산광역시',
      'signguCd': '26110',
      'signguNm': '중구',
      'rlteTatsCd': '5f3977fdbf61d8144813982e9c53609d',
      'rlteTatsNm': '영도관광사격장',
      'rlteRegnCd': '26',
      'rlteRegnNm': '부산광역시',
      'rlteSignguCd': '26200',
      'rlteSignguNm': '영도구',
      'rlteCtgryLclsNm': '관광지',
      'rlteCtgryMclsNm': '레저스포츠',
      'rlteCtgrySclsNm': '육상레저스포츠',
      'rlteRank': '1'}]},
   'numOfRows': 1,
   'pageNo': 1,
   'totalCount': 324}}}

In [ ]:
num_of_rows = 10000
params.update({"numOfRows":num_of_rows})

dfs = list()
for page in tqdm(range(1, total_count//num_of_rows + 2)):
    params.update({"pageNo":page})
    res = requests.get(URL, params= params)
    data = res.json()
    df = pd.DataFrame(data["response"]["body"]["items"]["item"])
    dfs.append(df[df["signguCode"].isin(busan_codes)])

 33%|███▎      | 79/237 [04:13<12:57,  4.92s/it]

In [43]:
df["signguNm"].value_counts()

중구     228
동구     228
서구     190
남구     152
북구     152
      ... 
경산시     38
의성군     38
청송군     38
영양군     38
성남시     38
Name: signguNm, Length: 239, dtype: int64

In [19]:
df = pd.concat(dfs,ignore_index=True)

In [22]:
df["areaNm"].value_counts()

서울특별시        8720
세종특별자치시      8720
강원특별자치도      8720
제주특별자치도      8720
경상남도         8720
경상북도         8720
충청남도         8720
충청북도         8720
경기도          8720
울산광역시        8720
대전광역시        8720
인천광역시        8720
대구광역시        8720
부산광역시        8720
전북특별자치도      8720
전라남도         8648
광주광역시        8648
전남광주통합특별시     141
Name: areaNm, dtype: int64